## 1. GPU Setup & Environment

In [ ]:
# Check GPU
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    # Enable TF32 for A100
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    print("✓ TF32 optimizations enabled")

In [ ]:
# Install required packages
!pip install -q transformers timm gradio opencv-python librosa soundfile scikit-learn tqdm 2>&1 | tail -5
print("✓ Packages installed")

In [ ]:
# Imports
import os
import numpy as np
import cv2
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt
from tqdm import tqdm
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from transformers import ViTForImageClassification, ViTImageProcessor
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import gradio as gr
import json
from datetime import datetime

print("✓ All imports successful")

## 2. Dataset Creation & Loading

In [ ]:
# Create sample FER2013 dataset
def create_sample_fer2013(root_dir='/content/data/raw/fer2013', n_train=50, n_test=10):
    """Create sample FER2013 dataset with dummy images"""
    emotions = ['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise']

    for split in ['train', 'test']:
        n_images = n_train if split == 'train' else n_test
        for emotion in emotions:
            emotion_dir = Path(root_dir) / split / emotion
            emotion_dir.mkdir(parents=True, exist_ok=True)

            for i in range(n_images):
                # Create random grayscale image
                img = np.random.randint(50, 200, (48, 48), dtype=np.uint8)
                img_path = emotion_dir / f'img_{i:04d}.jpg'
                cv2.imwrite(str(img_path), img)

    print(f"✓ Sample FER2013 created: {root_dir}")
    print(f"  Train: {n_train} images/emotion")
    print(f"  Test: {n_test} images/emotion")
    return root_dir

# Create dataset
fer2013_root = create_sample_fer2013()

In [ ]:
# Simple FER2013 Dataset class
class FER2013Dataset(Dataset):
    def __init__(self, root_dir, split='train', transform=None):
        self.root_dir = Path(root_dir)
        self.split = split
        self.transform = transform
        self.emotions = ['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise']
        self.emotion2idx = {e: i for i, e in enumerate(self.emotions)}

        self.samples = []
        self._load_samples()

    def _load_samples(self):
        split_dir = self.root_dir / self.split
        for emotion in self.emotions:
            emotion_dir = split_dir / emotion
            if emotion_dir.exists():
                for img_file in emotion_dir.glob('*.jpg'):
                    self.samples.append((str(img_file), self.emotion2idx[emotion]))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)

        if img is None:
            img = np.zeros((48, 48), dtype=np.uint8)

        img = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)

        if self.transform:
            img = self.transform(img)
        else:
            img = torch.from_numpy(img.transpose(2, 0, 1)).float() / 255.0

        return img, torch.tensor(label, dtype=torch.long)

print("✓ Dataset class defined")

In [ ]:
# Create dataloaders
transform_train = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                       std=[0.229, 0.224, 0.225])
])

transform_test = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                       std=[0.229, 0.224, 0.225])
])

train_dataset = FER2013Dataset(fer2013_root, split='train', transform=transform_train)
test_dataset = FER2013Dataset(fer2013_root, split='test', transform=transform_test)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=0)

print(f"✓ Train samples: {len(train_dataset)}")
print(f"✓ Test samples: {len(test_dataset)}")

## 3. ResNet-18 Baseline Training

In [ ]:
def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    for images, labels in tqdm(loader, desc="Training"):
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

def eval_epoch(model, loader, device):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for images, labels in tqdm(loader, desc="Evaluating"):
            images = images.to(device)
            outputs = model(images)
            preds = torch.argmax(outputs, dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.numpy())

    acc = accuracy_score(all_labels, all_preds)
    return acc, np.array(all_preds), np.array(all_labels)

print("✓ Training functions defined")

In [ ]:
# Train ResNet-18 baseline
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
num_epochs = 3
learning_rate = 0.001

# Initialize ResNet-18
resnet_model = models.resnet18(pretrained=True)
resnet_model.fc = nn.Linear(512, 7)  # 7 emotions
resnet_model = resnet_model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(resnet_model.parameters(), lr=learning_rate)

print("\n=== ResNet-18 Baseline Training ===")
for epoch in range(num_epochs):
    train_loss = train_epoch(resnet_model, train_loader, optimizer, criterion, device)
    test_acc, _, _ = eval_epoch(resnet_model, test_loader, device)
    print(f"Epoch {epoch+1}/{num_epochs} | Loss: {train_loss:.4f} | Test Acc: {test_acc:.4f}")

print("\n✓ ResNet-18 baseline training complete")

## 4. Vision Transformer Training

In [ ]:
# Load Vision Transformer
vit_model = ViTForImageClassification.from_pretrained(
    'google/vit-base-patch16-224-in21k',
    num_labels=7,
    ignore_mismatched_sizes=True
)
vit_model = vit_model.to(device)

# Optimizer
vit_optimizer = optim.AdamW(vit_model.parameters(), lr=5e-5)
vit_criterion = nn.CrossEntropyLoss()

print("✓ Vision Transformer model loaded")

In [ ]:
# Train Vision Transformer
num_vit_epochs = 3

print("\n=== Vision Transformer Training ===")
for epoch in range(num_vit_epochs):
    vit_model.train()
    total_loss = 0

    for images, labels in tqdm(train_loader, desc=f"ViT Epoch {epoch+1}"):
        images, labels = images.to(device), labels.to(device)
        vit_optimizer.zero_grad()
        outputs = vit_model(pixel_values=images)
        loss = vit_criterion(outputs.logits, labels)
        loss.backward()
        vit_optimizer.step()
        total_loss += loss.item()

    # Evaluate
    vit_model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(device)
            outputs = vit_model(pixel_values=images)
            preds = torch.argmax(outputs.logits, dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.numpy())

    test_acc = accuracy_score(all_labels, all_preds)
    print(f"Epoch {epoch+1}/{num_vit_epochs} | Loss: {total_loss/len(train_loader):.4f} | Test Acc: {test_acc:.4f}")

print("\n✓ Vision Transformer training complete")

## 5. Model Evaluation

In [ ]:
# Final evaluation
vit_model.eval()
all_preds, all_labels = [], []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        outputs = vit_model(pixel_values=images)
        preds = torch.argmax(outputs.logits, dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)

# Metrics
emotions = ['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise']
accuracy = accuracy_score(all_labels, all_preds)
precision = precision_score(all_labels, all_preds, average='weighted', zero_division=0)
recall = recall_score(all_labels, all_preds, average='weighted', zero_division=0)
f1 = f1_score(all_labels, all_preds, average='weighted', zero_division=0)

print("\n=== Vision Transformer Performance ===")
print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-Score:  {f1:.4f}")

In [ ]:
# Confusion matrix
cm = confusion_matrix(all_labels, all_preds)

plt.figure(figsize=(10, 8))
plt.imshow(cm, cmap='Blues', interpolation='nearest')
plt.title('Confusion Matrix - Vision Transformer')
plt.colorbar()
plt.xticks(range(7), emotions, rotation=45)
plt.yticks(range(7), emotions)
plt.xlabel('Predicted')
plt.ylabel('Actual')

# Add text annotations
for i in range(7):
    for j in range(7):
        plt.text(j, i, str(cm[i, j]), ha='center', va='center', color='white' if cm[i, j] > cm.max() / 2 else 'black')

plt.tight_layout()
plt.show()

print("✓ Confusion matrix displayed")

## 6. Save Model

In [ ]:
# Save model
model_dir = Path('/content/models/phase2')
model_dir.mkdir(parents=True, exist_ok=True)

model_path = model_dir / 'vit_emotion_model.pt'
torch.save({
    'model_state_dict': vit_model.state_dict(),
    'accuracy': accuracy,
    'timestamp': datetime.now().isoformat()
}, model_path)

print(f"✓ Model saved to {model_path}")

## 7. Interactive Gradio Demo

In [ ]:
def predict_emotion(image):
    """Predict emotion from image"""
    if image is None:
        return {emotion: 0 for emotion in emotions}

    # Prepare image
    img = cv2.cvtColor(np.array(image), cv2.COLOR_RGB2BGR)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # Transform
    img = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)
    img_tensor = transform_test(img).unsqueeze(0).to(device)

    # Predict
    with torch.no_grad():
        outputs = vit_model(pixel_values=img_tensor)
        logits = outputs.logits.cpu().numpy()[0]
        probs = np.exp(logits) / np.sum(np.exp(logits))

    # Return probabilities
    return {emotion: float(prob) for emotion, prob in zip(emotions, probs)}

# Create interface
demo = gr.Interface(
    fn=predict_emotion,
    inputs=gr.Image(type='pil', label='Upload Facial Image'),
    outputs=gr.Label(label='Emotion Probabilities', num_top_classes=7),
    title='Facial Emotion Recognition (Phase 2)',
    description='Upload a facial image to predict emotion using Vision Transformer'
)

print("✓ Gradio demo ready")

In [ ]:
# Launch demo
demo.launch(share=True)